In [1]:
#import the libraries
import pandas as pd
import numpy as np
import io

In [2]:
# 1. Simulating the raw CSV files
customers_csv = """CustomerID,Country,Subscription_Tier
C1,USA,Gold
C2,UK,Silver
C3,USA,Bronze
C4,Canada,Gold"""

products_csv = """ProductID,Category,Base_Price
P1,Electronics,500.0
P2,Apparel,50.0
P3,Electronics,800.0
P4,Home,120.0"""

# Notice the missing discount values (blank spaces) for T002 and T004!
transactions_csv = """OrderID,CustomerID,ProductID,Quantity,Discount_Pct
T001,C1,P1,2,0.10
T002,C2,P2,3,
T003,C1,P3,1,0.20
T004,C3,P1,1,
T005,C99,P4,2,0.05
T006,C4,P2,1,0.15"""

# 2. Loading them into pandas (This mimics pd.read_csv('file.csv'))
df_customers = pd.read_csv(io.StringIO(customers_csv))
df_products = pd.read_csv(io.StringIO(products_csv))
df_transactions = pd.read_csv(io.StringIO(transactions_csv))

In [3]:
df_customers

,CustomerID,Country,Subscription_Tier
0,C1,USA,Gold
1,C2,UK,Silver
2,C3,USA,Bronze
3,C4,Canada,Gold


In [4]:
df_products

,ProductID,Category,Base_Price
0,P1,Electronics,500.0
1,P2,Apparel,50.0
2,P3,Electronics,800.0
3,P4,Home,120.0


In [6]:
df_transactions

,OrderID,CustomerID,ProductID,Quantity,Discount_Pct
0,T001,C1,P1,2,0.10
1,T002,C2,P2,3,NaN
2,T003,C1,P3,1,0.20
3,T004,C3,P1,1,NaN
4,T005,C99,P4,2,0.05
5,T006,C4,P2,1,0.15


In [7]:
#Clean up
df_transactions['Discount_Pct']=df_transactions['Discount_Pct'].fillna(0)

In [9]:
df_transactions

,OrderID,CustomerID,ProductID,Quantity,Discount_Pct
0,T001,C1,P1,2,0.10
1,T002,C2,P2,3,0.00
2,T003,C1,P3,1,0.20
3,T004,C3,P1,1,0.00
4,T005,C99,P4,2,0.05
5,T006,C4,P2,1,0.15


In [ ]:
#Merging the dataframes to create a master dataframe
master_df=pd.merge(df_transactions,df_customers, on='CustomerID', how='left')
master_df=pd.merge(master_df,df_products, on='ProductID', how='left')

In [13]:
master_df

,OrderID,CustomerID,ProductID,Quantity,Discount_Pct,Country,Subscription_Tier,Category,Base_Price
0,T001,C1,P1,2,0.10,USA,Gold,Electronics,500.0
1,T002,C2,P2,3,0.00,UK,Silver,Apparel,50.0
2,T003,C1,P3,1,0.20,USA,Gold,Electronics,800.0
3,T004,C3,P1,1,0.00,USA,Bronze,Electronics,500.0
4,T005,C99,P4,2,0.05,NaN,NaN,Home,120.0
5,T006,C4,P2,1,0.15,Canada,Gold,Apparel,50.0


In [20]:
master_df['Revenue']=master_df['Quantity'] * master_df['Base_Price'] * (1 - master_df['Discount_Pct'])

In [21]:
master_df

,OrderID,CustomerID,ProductID,Quantity,Discount_Pct,Country,Subscription_Tier,Category,Base_Price,Revenue
0,T001,C1,P1,2,0.10,USA,Gold,Electronics,500.0,900.0
1,T002,C2,P2,3,0.00,UK,Silver,Apparel,50.0,150.0
2,T003,C1,P3,1,0.20,USA,Gold,Electronics,800.0,640.0
3,T004,C3,P1,1,0.00,USA,Bronze,Electronics,500.0,500.0
4,T005,C99,P4,2,0.05,NaN,NaN,Home,120.0,228.0
5,T006,C4,P2,1,0.15,Canada,Gold,Apparel,50.0,42.5


In [22]:
Summary_df = master_df.groupby(['Country', 'Category']).agg({
    'Revenue':'sum',
    'OrderID':'count'
}).reset_index()


In [24]:
print(Summary_df)

  Country     Category  Revenue  OrderID
0  Canada      Apparel     42.5        1
1      UK      Apparel    150.0        1
2     USA  Electronics   2040.0        3
